In [ ]:

from pathlib import Path
import sys
import yaml

def find_project_root(start=None):
    start = Path(start or Path.cwd()).resolve()

    for candidate in [start, *start.parents]:
        if (candidate / "config.yaml").exists():
            return candidate

    raise FileNotFoundError(
        "No se encontró config.yaml. "
        "Ejecute Jupyter desde la carpeta del proyecto o notebooks/."
    )

ROOT = find_project_root()

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

cfg = yaml.safe_load(
    (ROOT / "config.yaml").read_text(encoding="utf-8")
)

print("Proyecto:", ROOT)


## 1. Cargar data

In [ ]:
import pandas as pd

manifest = pd.read_csv(
    ROOT / "data/manifests/dataset_manifest.csv"
)

print("Total de imágenes:", len(manifest))
display(manifest.head(10))

## 2. Cantidad de imágenes por grupo

In [ ]:
group_counts = (
    manifest.groupby("group")
    .size()
    .rename("n")
    .reset_index()
)

display(group_counts)

## 3. Estadísticas de las imágenes usadas

In [ ]:
dimension_stats = (
    manifest.groupby("group")
    [["width","height","megapixels","size_mb"]]
    .agg(["count","min","median","mean","max"])
    .round(3)
)

display(dimension_stats)

## 4. Estadísticas de calidad

In [ ]:
image_stats = (
    manifest.groupby("group")
    [["brightness_mean","brightness_std"]]
    .agg(["mean","std","min","max"])
    .round(3)
)

display(image_stats)

## 5. Verificación de duplicados

In [ ]:
exact = pd.read_csv(
    ROOT / "data/manifests/exact_duplicates.csv"
)

near = pd.read_csv(
    ROOT / "data/manifests/near_duplicate_audit.csv"
)

print("Duplicados exactos:", len(exact))

display(
    near.sort_values("ahash_hamming")
    .head(15)
)

## 6. Gráficas de auditoría

In [ ]:
print("Las figuras correspondientes se generan y exportan al final de P1 en la carpeta results/.")

## 7. Mosaicos representativos

In [ ]:
print("Las figuras correspondientes se generan y exportan al final de P1 en la carpeta results/.")

## Exportación de auditoría

Esta sección genera y guarda las salidas de auditoría que se utilizan en el proyecto.


In [ ]:
from PIL import Image, ImageOps
import numpy as np
import matplotlib.pyplot as plt
import json, random

AUDIT_DIR = ROOT / "results/audit"
MOSAIC_DIR = ROOT / "results/mosaics"
AUDIT_DIR.mkdir(parents=True, exist_ok=True)
MOSAIC_DIR.mkdir(parents=True, exist_ok=True)

# Tablas de auditoría
_group_counts = manifest.groupby("group").size().rename("n").reset_index()
_group_counts.to_csv(AUDIT_DIR / "group_counts.csv", index=False)

_dim = manifest.groupby("group")[["width","height","megapixels","size_mb"]].agg(["count","min","median","mean","max"]).round(4)
_dim.to_csv(AUDIT_DIR / "dimension_stats.csv")

_img = manifest.groupby("group")[["brightness_mean","brightness_std"]].agg(["mean","std","min","max"]).round(4)
_img.to_csv(AUDIT_DIR / "image_stats.csv")

summary = {
    "total_images": int(len(manifest)),
    "groups": {str(k): int(v) for k,v in manifest.groupby("group").size().items()},
    "exact_duplicate_pairs": int(len(exact)),
    "near_duplicate_pairs": int(len(near)),
}
(AUDIT_DIR / "audit_summary_notebook.json").write_text(json.dumps(summary, indent=2, ensure_ascii=False), encoding="utf-8")

# Figuras de auditoría generadas desde el manifiesto
plots = [
    ("dimensions_by_group_notebook.png", "megapixels", "Megapíxeles", "Dimensiones por grupo"),
    ("brightness_by_group_notebook.png", "brightness_mean", "Brillo medio", "Brillo por grupo"),
    ("contrast_by_group_notebook.png", "brightness_std", "Desviación de intensidad", "Contraste por grupo"),
]
for fname, col, ylabel, title in plots:
    fig, ax = plt.subplots(figsize=(7.2,4.2))
    groups = list(manifest["group"].dropna().unique())
    vals = [manifest.loc[manifest.group==g, col].dropna().values for g in groups]
    ax.boxplot(vals, labels=groups, showfliers=False)
    ax.set_ylabel(ylabel); ax.set_title(title); ax.grid(axis="y", alpha=.2)
    plt.xticks(rotation=20, ha="right")
    fig.tight_layout(); fig.savefig(AUDIT_DIR/fname, dpi=220, bbox_inches="tight"); plt.close(fig)

# Mosaicos deterministas
random.seed(int(cfg["project"]["seed"]))

def save_mosaic(paths, out, title, n=16, cols=4):
    out=Path(out)
    if out.exists():
        print('Reutilizando mosaico:', out.name)
        return
    paths=[Path(p) for p in paths if Path(p).exists()]
    if not paths: return
    picks=paths if len(paths)<=n else random.sample(paths,n)
    rows=int(np.ceil(len(picks)/cols))
    fig, axs=plt.subplots(rows, cols, figsize=(cols*2.3, rows*2.3))
    axs=np.atleast_1d(axs).ravel()
    for ax in axs: ax.axis('off')
    for ax,p in zip(axs,picks):
        im=ImageOps.exif_transpose(Image.open(p)).convert('RGB')
        im.thumbnail((900,900))
        ax.imshow(im); ax.set_title(p.name, fontsize=7); ax.axis('off')
    fig.suptitle(title, fontsize=11); fig.tight_layout()
    fig.savefig(out, dpi=180, bbox_inches='tight'); plt.close(fig)

split_df = pd.read_csv(ROOT/cfg["data"]["supervised_split"])
train_paths=[ROOT/p for p in split_df.loc[split_df.split=='train','path'].astype(str)]
field_paths=sorted((ROOT/'pictures/anomaly_field').glob('*'))
test_paths=sorted((ROOT/'pictures/test').glob('*'))
save_mosaic(train_paths, MOSAIC_DIR/'train_random_mosaic_notebook.png', 'Muestra del entrenamiento')
save_mosaic(field_paths, MOSAIC_DIR/'anomaly_field_mosaic_notebook.png', 'Fotografías de campo')
save_mosaic(test_paths, MOSAIC_DIR/'test_random_mosaic_notebook.png', 'Muestra de prueba de campo')

print('P1 exportó auditoría y mosaicos en results/audit y results/mosaics.')


In [ ]:
from pathlib import Path
import shutil
PAPER_OUT = ROOT / "results/paper"
PAPER_OUT.mkdir(parents=True, exist_ok=True)
legacy = ROOT / "paper/figures"
for stem in ["fig1_metodologia", "fig2_arquitectura", "fig8_dataset"]:
    for ext in ["png", "pdf", "svg"]:
        src = legacy / f"{stem}.{ext}"
        if src.exists():
            shutil.copy2(src, PAPER_OUT / src.name)
print("P1 publicó metodología, arquitectura y mosaico de dataset en results/paper.")
